###  Company DCF Valuation

Explanation: In this project, I will make a set of functions which give me information about publicly listed companies on the stock market. The most substantial of which will be a DCF valuation, which seeks to assign a value to all of the future cash that a company will generate. A DCF valuation is one method of estimating the overall value of a company.

This will all be aggregated in one function called stock_info, which provides me with basic information the stock, alongside my DCF valuation.

The DCF formula is as follows:

### Discounted Cash Flow (DCF) Formula

The value of a company as the sum of its discounted future cash flows:

$$DCF = \sum_{t=1}^{n} \frac{FCF_t}{(1 + r)^t} + \frac{TV}{(1 + r)^n}$$


Where:
- \(DCF\) = Discounted Cash Flow 
- \(FCF_t\) = Free Cash Flow at time \(t\), recorded annually
- \(r\) = Discount rate (WACC or required rate of return)
- \(n\) = Number of periods (projection years)
- \(TV\) = Terminal Value

### Terminal Value (TV)
The terminal value can be calculated using the Gordon Growth Model:


$$TV = \frac{FCF_{n+1}}{r - g}$$


Where:
- \(FCF_{n+1}\) = Free Cash Flow in the year after the last projected year
- \(r\) = Discount rate
- \(g\) = Perpetual growth rate

### Discount Rate (WACC)


$$ WACC = \frac{E}{E + D} \cdot Re + \frac{D}{E + D} \cdot Rd \cdot (1 - Tc) $$


Where:
- \(WACC\): Weighted Average Cost of Capital
- \(E\): Market value of equity
- \(D\): Market value of debt
- \(Re\): Cost of equity
- \(Rd\): Cost of debt
- \(Tc\): Corporate tax rate<br><br>


### Procedure
Essentially, for this project, I will be pulling the necessary variables from the yfinance library to complete the above formulas and produce an estimation of DCF valuation. 

In order to complete this, I did the following steps:

     
   
1. Construct the pieces for my DCF function individually;
   - WACC Weighted average cost of capital
     the inputs for this function are relatively simple, and are either directly pulled from yfinance or are basic ratios
   - FCF (Free Cash Flow) Values
   - I did a very basic linear regression using the previous four years of cash flow data to forecast the next four years of FCF values.
   - Terminal value can be calculated using these predictions, plus the prediction that GDP will grow by about 2.5% each year. <br><br>
     
2. Put these two functions into my dcf function
   
3. Built a basic version of the stock_info function, which returned basic financial information about a company based on its stock market ticker symbol. **The input for the stock_info function should be the ticker symbol (1-4 letters) as an string.**
   - I used the yfinance library, which has updated information about the stock market, to produce this data.
   - The only value it was missing was the DCF valuation. <br><br>
   


In [3]:
#importing relevant libraries
#pandas and numpy for data analysis 
#yfinance for financial information
#sklearn for regressions

!pip install yfinance
import pandas as pd
import numpy as np
import yfinance as yf
from sklearn.linear_model import LinearRegression
!pip install pytest


In [4]:
#Step 1.1
#Finding the value for r (wacc) using the formula described aboe

def wacc(tick):

    try:

        ticker = yf.Ticker(tick)
    
        #Determine market value of equity and debt
        market_value_of_equity = ticker.info['sharesOutstanding'] * ticker.info['currentPrice']
        market_value_of_debt = ticker.balance_sheet.loc["Total Debt"].iloc[0]
    
        #Determine tax rate (directly available on yf)
        tax_rate = ticker.financials.loc['Tax Rate For Calcs'].iloc[0]
        
        #Determine cost of debt
        interest_expense = ticker.financials.loc['Interest Expense'].iloc[0]
        total_debt = ticker.balance_sheet.loc["Total Debt"].iloc[0]
        cost_of_debt = interest_expense / total_debt
    
        #Determine cost of equity
        risk_free_rate = (yf.Ticker("^TNX").history(period="1d")["Close"].iloc[-1])/100
        beta = ticker.info['beta']
    
        market_ror_used = 0.08
    
        cost_of_equity = risk_free_rate + (beta *(market_ror_used - risk_free_rate))
        
        #This is EXTRANEOUS code which I used to try to calculate the market ror. Currently, it does not accurately reflect 
        #consensus values for it, so I will use the institutionally accepted rate of 8%, as shown above.
        market_price_10ya = yf.Ticker("^GSPC").history(period="10y")["Close"].iloc[0] 
        market_price_today = yf.Ticker("^GSPC").history(period='1d')["Close"].iloc[-1]
        year_10_change = market_price_today / market_price_10ya
        market_ror_calculated = year_10_change**(1/10) - 1
    
        #Weigh cost of capital and cost of equity according to capital structure
        weighted_cost_of_equity = (market_value_of_equity / (market_value_of_equity + market_value_of_debt)) * cost_of_equity
        weighted_cost_of_debt = (market_value_of_debt / (market_value_of_equity + market_value_of_debt)) * cost_of_debt * (1 - tax_rate)
        
        #Sum weighted averages of C.O. Capital and C.O. Debt
        wacc = weighted_cost_of_equity + weighted_cost_of_debt
    
        return wacc

    except Exception as e:

        print(f"Error calculating wacc for {tick}: {e}")

        return np.nan




In [5]:
#STEP 1.2
#Finding FCF values and TV value from formulas above. 
#These are components that will feed into the dcf formula.
#created a basic regression using the previous four years of FCF

def fcfs(tick):

    try:
    
        ticker = yf.Ticker(tick)
    
        #arrange x (year count) and y (FCF value) into an array, run a linear regression on that array
        x = np.array([1, 2, 3, 4]).reshape(-1, 1)
        y = np.array([
            (ticker.cashflow.loc["Free Cash Flow"].iloc[3]),
            (ticker.cashflow.loc["Free Cash Flow"].iloc[2]),
            (ticker.cashflow.loc["Free Cash Flow"].iloc[1]),
            (ticker.cashflow.loc["Free Cash Flow"].iloc[0])
            ])
    
        model = LinearRegression().fit(x, y)
    
        #pull slope and intercept from FCF regression
        intercept = model.intercept_
        slope = model.coef_[0]
    
        sum_of_annual_fcf = (ticker.cashflow.loc["Free Cash Flow"].iloc[3]) + \
                            (ticker.cashflow.loc["Free Cash Flow"].iloc[2]) + \
                            (ticker.cashflow.loc["Free Cash Flow"].iloc[1]) + \
                            (ticker.cashflow.loc["Free Cash Flow"].iloc[0])
        
        #calculate growth rate of a percentage by using slope
        growth_rate_fcf = 1 + ((4*slope) / (sum_of_annual_fcf))
    
        #utilize that growth rate to project free cash flow for future years. terminal value captures all of the fcfs into 
        #perpetuity.
        long_term_growth_rate = 0.025
        fcf1 = growth_rate_fcf * ticker.cashflow.loc["Free Cash Flow"].iloc[0]
        fcf2 = growth_rate_fcf * fcf1 
        fcf3 = growth_rate_fcf * fcf2
        fcf4 = growth_rate_fcf * fcf3
        terminal_value = (fcf4 * growth_rate_fcf) / (wacc('msft') - long_term_growth_rate)
    
        return (fcf1, fcf2, fcf3, fcf4, terminal_value)

    except Exception as e:
    
        print(f"Error calculating fcfs for {tick}: {e}")

        return np.nan

#Step 2: put WACC value and DCF values into dcf formula 


In [7]:
#Step 2: put WACC value and DCF values into dcf formula 

def dcf(tick):

    try: 
        #Pull out output of fcf function and assign to relevant variable names
        fcf_value_list = fcfs(tick)
        fcf1, fcf2, fcf3, fcf4, terminal_value =    fcf_value_list[0], \
                                                    fcf_value_list[1], \
                                                    fcf_value_list[2], \
                                                    fcf_value_list[3], \
                                                    fcf_value_list[4]
        
        #assemble more variables using formulas created earlier
        r = wacc(tick)
        fcf_years = (fcf1, fcf2, fcf3, fcf4)
        dcf = 0
    
        #sum discounted cash flows for four years
        for i in range(len(fcf_years)):
    
            dcf += fcf_years[i]/((1 + r)**(1+i))
    
        #calculate and add present value of terminal value (pv_tv)
    
        pv_tv = terminal_value / ((1 + r)**4)
    
        dcf += pv_tv
    
        #option for output, returns output with commas (not used as I want an int or float output for testing)
        formated = f"{dcf:,}"
    
        return dcf

    except Exception as e:

        print(f"Error calculating DCF for {tick}: {e}")

        return np.nan


In [8]:
dcf('aapl')

nan

In [9]:
#Step 3: Creating a basic stock info function
#Takes ticker symbol (1-4 letters, as int) and returns financial information
#also references my dcf function

def stock_info(tick):

    #pulling values from yf library
    stock_info = yf.Ticker(tick)
    current_price = yf.download(tick, start='2024-12-2', end='2024-12-3')
    company_name = stock_info.info['longName']
    description = stock_info.info['longBusinessSummary']
    market_cap = str(stock_info.info['marketCap'])
    price_to_equity = str(stock_info.info['trailingPE'])
    current_price = str(stock_info.info['currentPrice'])
    year_high = str(stock_info.info['fiftyTwoWeekHigh'])
    year_low = str(stock_info.info['fiftyTwoWeekLow'])

    #organize outputs into string output for readability
    output = (
    'Company Name: ' + company_name + '\n' + '\n' +
    description + '\n' + '\n' +
    'Market Capitalization: ' + market_cap + '\n' + '\n' +
    'Price to Equity Ratio: ' + price_to_equity + '\n' + '\n' +
    'Current Price: ' + current_price + '\n' + '\n' +
    '52 Week High: ' + year_high + '\n' + '\n' +
    '52 Week Low: ' + year_low +'\n' + '\n' +
    'DCF Valuation: ' + str(dcf(tick)) 
    )

    print (output)


    

In [22]:
stock_info('nvda')

[*********************100%***********************]  1 of 1 completed


Company Name: NVIDIA Corporation

NVIDIA Corporation provides graphics and compute and networking solutions in the United States, Taiwan, China, Hong Kong, and internationally. The Graphics segment offers GeForce GPUs for gaming and PCs, the GeForce NOW game streaming service and related infrastructure, and solutions for gaming platforms; Quadro/NVIDIA RTX GPUs for enterprise workstation graphics; virtual GPU or vGPU software for cloud-based visual and virtual computing; automotive platforms for infotainment systems; and Omniverse software for building and operating metaverse and 3D internet applications. The Compute & Networking segment comprises Data Center computing platforms and end-to-end networking platforms, including Quantum for InfiniBand and Spectrum for Ethernet; NVIDIA DRIVE automated-driving platform and automotive development agreements; Jetson robotics and other embedded platforms; NVIDIA AI Enterprise and other software; and DGX Cloud software and services. The company'

In [11]:
#Step 3.1: A short informal test to see how large my error is. I did not make this an empirically sound test because
#my dcf valuation is not meant to be reflected in market capitalization amounts. In other words, my model won't and shouldn't
#match stock market info perfectly. As long as the values are very roughly in the same ballpark, that shows that the model is 
#possibly right. (essentially checking that it isn't definitely wrong)

test_list = ['F', 'KR', 'MMM', 'AOS','ACN','AMD','NVDA','MSFT','AMZN']

error_percentage_list = []

for ticker in test_list:
    
    error = abs(dcf(ticker) - yf.Ticker(ticker).info['marketCap']) / yf.Ticker(ticker).info['marketCap']

    error_percentage_list.append(error)

average_error = (sum(error_percentage_list)) / len(error_percentage_list)

print(average_error)

#This test returns an error value of 39.8% which means that the numbers are typically of very roughly, similar size, so my model
#is possibly usable. However, there would need to be a lot more work done to get a model that would match market capitalization values
#well. 


9.11891205345241


In [12]:
!pytest

============================= test session starts ==============================
platform darwin -- Python 3.12.7, pytest-7.4.4, pluggy-1.0.0
rootdir: /Users/kyabetsu/Desktop/Python files/Final Project
plugins: anyio-4.2.0
collected 0 items / 1 error                                                    

==================================== ERRORS ====================================
___________________ ERROR collecting Test/test_functions.py ____________________
ImportError while importing test module '/Users/kyabetsu/Desktop/Python files/Final Project/Test/test_functions.py'.
Hint: make sure your test modules/packages have valid Python names.
Traceback:
/opt/anaconda3/lib/python3.12/importlib/__init__.py:90: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
Test/test_functions.py:9: in <module>
    from final_project import wacc, fcfs, dcf
E   ModuleNotFoundError: No module named 'final_project'
=========================== short test summary info ========

## Challenges
Why this project was challenging to me: <br>

- Implementing Discounted Cash Flow (DCF) analysis required understanding financial concepts like Weighted Average Cost of Capital (WACC), terminal value calculations, and Free Cash Flow (FCF) forecasting. For these concepts, I had to do outside learning and studying on my own, prior to being able to create a model in code.


- Using the yfinance library to fetch stock market data introduced challenges due to incomplete, inconsistent, or missing data.
I had to properly reference different types of data and pull them in ways such that it would not make the program unreliable (although I am sure that thre is a lot more room for improving the reliability of the code.

- I also used the scikit-learn library to perform linear regression for forecasting, which involved reshaping data arrays and interpreting regression model outputs—concepts not covered in class.

